In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

"wget" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


"wget" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os

LLM_MODEL = "MiniMax-M2.7"

load_dotenv(override=True)

api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")
if not api_key or not base_url:
    raise SystemExit(
        "Missing env vars. Set OPENAI_API_KEY and OPENAI_BASE_URL in .env."
    )

openai_client = OpenAI(api_key=api_key, base_url=base_url)

In [4]:
from agentic_rag.ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [12]:
from agentic_rag.rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    model=LLM_MODEL
)

In [8]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

## Running Ollama Locally

To run Ollama locally, follow these steps:

### 1. Install Ollama

Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:

- **macOS**: Download the `.pkg` and install it
- **Windows**: Download the `.msi` and install it
- **Linux**: Run the following command in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

### 2. Run a Model

Open a terminal and type:
```bash
ollama run llama3
```

This command will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a chat-like interface where you can type questions

### 3. Test the Local Server

Run the following command to verify the server is running:
```bash
curl http://localhost:11434
```

You should receive a response similar to:
```json
{"models": [...]}
```

### 4. Install the Python Client (Optional)

```bash
pip install ollama
```

### 5. Example Python Usage

```python
import ollama

response = ollama.chat(
    model='l

In [9]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

Based on the provided CONTEXT, there is no information about how to run Ollama locally. The CONTEXT does not contain any questions or answers specifically about Ollama.

The only related information available is about running the course locally in general:

> "You can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module."

If you have questions about Ollama specifically, they are not covered in this FAQ database.


In [13]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model=LLM_MODEL,
    input=messages,
)

response.output_text

'Sure thing! I’d be happy to help you get started. To give you the most accurate steps, could you let me know a little more about the course you’re referring to? For example:\n\n| What I need to know | Why it helps |\n|----------------------|--------------|\n| **Course title or subject** (e.g., “Intro to Machine Learning,” “Creative Writing”) | Different platforms have different enrollment procedures. |\n| **Where it’s offered** (university, online platform like Coursera/edX/Udemy, corporate training site, etc.) | The registration process varies—sometimes you create an account, sometimes you need a university login. |\n| **Delivery format** (in‑person, live‑online, self‑paced) | This determines whether you need to sign up for a specific session or can start anytime. |\n| **Any prerequisites** (e.g., basic Python knowledge, a specific degree) | Some courses require you to verify certain skills before enrolling. |\n| **Enrollment period & deadline** (open now, closes on a specific date) 

In [23]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [24]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [25]:
response = openai_client.responses.create(
    model=LLM_MODEL,
    input=messages,
    tools=[search_tool]
)

In [27]:
function_calls = [item for item in response.output if getattr(item, 'type', None) == 'function_call']

In [38]:
if function_calls:
    call = function_calls[0]
    import json
    args = json.loads(call.arguments)
    print("Function arguments:", args)
    
    # Run the search
    results = search(**args)
    # ... process results as usual ...
else:
    # No tool was called; print the assistant's text response instead
    print("No tool call generated. Assistant response:")
    print(response.output_text)

Function arguments: {'query': 'join course enrollment'}


In [39]:
len(response.output)

2

In [41]:
results = search(**args)

In [42]:
result_json = json.dumps(results, indent=2)

In [43]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [44]:
messages.append(call)

In [45]:
messages.append(function_call_output)

In [46]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query": "join course enrollment"}', call_id='call_function_bup8hcud82o5_1', name='search', type='function_call', id='0686678617b35db6a3abd4a8d1b0b561_fc_0', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_function_bup8hcud82o5_1',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027.",\n    "doc_id": "bd31146b0e"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section"

In [47]:
response = openai_client.responses.create(
    model=LLM_MODEL,
    input=messages,
    tools=[search_tool]
)

In [48]:
print(response.output_text)

Yes, you can still join! The course materials are available and you can start learning at your own pace.

However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. The system tracks deadlines through the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [49]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(930, 146)

In [50]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.2   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 1.2  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00017


In [51]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [52]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [54]:
response = openai_client.responses.create(
    model=LLM_MODEL,
    input=messages,
    tools=[search_tool]
)


In [55]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
I'd be happy to help you learn about joining the course! Let me search for information about course enrollment.

function_call: search {"query": "join course enrollment registration"}


In [56]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='0686698920b6f4b3cad2a7106c26bc5c_rs', summary=[], type='reasoning', content=[Content(text='The user is asking about joining a course. This seems like a general question about course enrollment or registration. I should search for information about this in the course FAQ database.\n\nLet me search using keywords related to joining/enrolling in a course.', type='reasoning_text')], encr

In [57]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model=LLM_MODEL,
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break


iteration #1...
ASSISTANT:
I'll help you find information about joining the course. Let me search for relevant FAQ entries.

function_call: search {"query": "join course enrollment registration"}
iteration #2...
ASSISTANT:
I found relevant information. Let me search for additional details about late enrollment and certificate requirements.

function_call: search {"query": "late enrollment certificate deadline"}
iteration #3...
ASSISTANT:
Based on the course FAQ, here's what you need to know:

## Yes, You Can Join!

**Good news:** You can absolutely join the course even if you've just discovered it.

### Key Points:

1. **Registration isn't mandatory:** You're accepted automatically when you register. You can even start learning and submitting homework without registering - it's mainly to gauge interest before the start date.

2. **Self-paced learning:** You can start whenever you want! All videos and GitHub materials are available, and you can work through them at your own pace.

3. **

In [59]:
def agent_loop(instructions, question, model) -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [60]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [61]:
result = agent_loop(instructions, question, model=LLM_MODEL)

iteration #1...
ASSISTANT:
I'd be happy to help you learn more about joining the course! Let me search for information about course enrollment and registration.

function_call: search {"query": "join course enrollment registration"}
iteration #2...
ASSISTANT:
Based on my search, great news! **Yes, you can still join the course!**

Here's what I found:

1. **You can join anytime** - The course materials (videos and GitHub materials) are available, and you can start learning whenever you want.

2. **Registration is flexible** - Registration is mainly to gauge interest before the start date. You don't need to wait for a confirmation email — you're accepted! You can even start submitting homework while the registration form is still open.

3. **For certificates** - If you want to receive a certificate, you need to submit your project while submissions are still being accepted. Note that certificates are only awarded for completing the course with a "live" cohort (not in self-paced mode), a

In [62]:
result

'Based on my search, great news! **Yes, you can still join the course!**\n\nHere\'s what I found:\n\n1. **You can join anytime** - The course materials (videos and GitHub materials) are available, and you can start learning whenever you want.\n\n2. **Registration is flexible** - Registration is mainly to gauge interest before the start date. You don\'t need to wait for a confirmation email — you\'re accepted! You can even start submitting homework while the registration form is still open.\n\n3. **For certificates** - If you want to receive a certificate, you need to submit your project while submissions are still being accepted. Note that certificates are only awarded for completing the course with a "live" cohort (not in self-paced mode), and you\'ll also need to peer-review 3 capstone projects from other students.\n\n**How to get started:**\n- Check the course documentation\n- Visit the course management platform for deadlines\n- Follow the typical workflow: watch lesson videos → wo

In [64]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question, model=LLM_MODEL)

iteration #1...
ASSISTANT:
I should search the FAQ database to see if "queen gambit" is mentioned in relation to the course.


function_call: search {"query": "queen gambit"}
iteration #2...
ASSISTANT:
I'm sorry, but I couldn't find any information about "queen gambit" in the course FAQ. This appears to be a question about chess (which is a chess opening), rather than about the course or its logistics.

As a course teaching assistant, I'm only able to answer questions related to the course or its logistics. Could you please ask a question that is related to the course?

Is there anything else about the course you'd like to explore?


In [65]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [66]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [67]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [68]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [69]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [70]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [71]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=LLM_MODEL, client=openai_client)
)

In [72]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [73]:
result.cost

CostInfo(input_cost=Decimal('0.0006048'), output_cost=Decimal('0.0008112'), total_cost=Decimal('0.0014160'))

In [74]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='06866aab50c8cede4db6585407d3f841_rs', summary=[], type='reasoning', content=[

In [75]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [ ]:
runner.run();